In [ ]:
# Standard libraries
import math
import random

# Hide deprecation warnings
import warnings
warnings.filterwarnings('ignore')

# PyTorch core
import torch
import torch.nn as nn
import torch.optim as optim

# For processing data
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

# BPE tokenizer
import tiktoken

# Training utils
from torch.nn.utils import clip_grad_norm_
from torch.optim.lr_scheduler import CosineAnnealingLR

# Progress bar
from tqdm import tqdm

In [ ]:
# Load WikiText dataset

train_data = load_dataset("wikitext", "wikitext-103-v1", split="train") # Training subset
val_data   = load_dataset("wikitext", "wikitext-103-v1", split="validation") # Validation subset

In [ ]:
# Training subset example
print(train_data['text'][807])

In [ ]:
# Unwanted training subset example
print(train_data['text'][804:807])

In [ ]:
# UNK marker
print(train_data['text'][80])

In [ ]:
def clean_text(dataset):
    # EOS token
    EOS = "<|endoftext|>"

    cleaned = []
    for text in dataset["text"]:
        # Strip surrounding whitespace
        line = text.strip()

        # Keep only non-empty lines that aren't headings (e.g., "= = Definitions = =")
        if line and not line.startswith("="):
            # Remove <unk> markers and normalize whitespace
            cleaned.append(" ".join(line.replace("<unk>", "").split()))

    # Join all entries into a string, separated by EOS token
    return EOS.join(cleaned) + EOS

# Clean training and validation text
training_text = clean_text(train_data)
validation_text = clean_text(val_data)

In [ ]:
# Print a subset of 'training_text'
print(training_text[3457:3700])

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

vocab_size = tokenizer.n_vocab
print(f"Vocabulary size: {vocab_size}")

In [ ]:
class TextDataset(Dataset):
    def __init__(self, text, tokenizer, max_seq_length):
        # Convert text to token IDs
        self.tokens = tokenizer.encode(text, allowed_special="all")

        # Maximum length of each training sequence
        self.max_seq_length = max_seq_length

    # Get number of valid training sequences
    def __len__(self):
        num_sequences = (len(self.tokens) - 1) // self.max_seq_length
        return num_sequences

    # Get an input sequence and targets
    def __getitem__(self, idx):
        # Start index of the sequence
        start = idx * self.max_seq_length

        # End index of the sequence
        end = start + self.max_seq_length

        # Input token sequence
        input_ids = torch.tensor(self.tokens[start:end], dtype=torch.long)

        # Next-token targets/ labels (shifted by one character)
        target_ids = torch.tensor(self.tokens[start+1:end+1], dtype=torch.long)

        return input_ids, target_ids

In [ ]:
MAX_SEQ_LENGTH = 128

train_dataset = TextDataset(training_text, tokenizer, MAX_SEQ_LENGTH)
val_dataset = TextDataset(validation_text, tokenizer, MAX_SEQ_LENGTH)

In [ ]:
print(f"Number of training sequences: {len(train_dataset):,}")
print(f"Number of validation sequences: {len(val_dataset):,}")

In [ ]:
input, target = train_dataset[16]

print("Input IDs:\n", input)
print("\nTarget IDs:\n", target)

print("\nDecoded Input:\n", tokenizer.decode(input.tolist()))
print("\nDecoded Target:\n", tokenizer.decode(target.tolist()))

In [ ]:
# Define batch size
BATCH_SIZE = 32

# Create training and validation DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
print(f"Number of training batches: {len(train_loader):,}") # Number of training sequences / batch_size
print(f"Number of validation batches: {len(val_loader):,}") # Number of validation sequences / batch_size

In [ ]:
# Grouped Query Attention (GQA) with Causal Masking
class GroupedQueryAttention(nn.Module):
    def __init__(self, embedding_dim, num_heads, num_groups, max_seq_length):
        super().__init__()

        # Check if embedding_dim is divisible by num_heads
        assert embedding_dim % num_heads == 0, "embedding_dim must be divisible by num_heads"

        # Check if num_heads is divisible by num_groups
        # (Each group must be shared by the same number of heads)
        assert num_heads % num_groups == 0, "num_heads must be divisible by num_groups"

        # Embedding dimension
        self.embedding_dim = embedding_dim

        # Number of total query heads
        self.num_heads = num_heads

        # Dimension of each head
        self.head_dim = embedding_dim // num_heads

        # Number of KV groups
        self.num_groups = num_groups

        # Number of query heads per KV group
        self.group_size = num_heads // num_groups

        # Linear projection matrix for query
        self.W_q = nn.Linear(embedding_dim, embedding_dim, bias=False)

        # Linear projection matrices for key and value
        self.W_k = nn.Linear(embedding_dim, num_groups * self.head_dim, bias=False)
        self.W_v = nn.Linear(embedding_dim, num_groups * self.head_dim, bias=False)

        # Linear projection matrix to produce final output
        self.W_o = nn.Linear(embedding_dim, embedding_dim, bias=False)

        # Build the causal mask once at init instead of every forward pass
        mask = torch.tril(torch.ones(max_seq_length, max_seq_length))

        # Add batch_size and num_heads dimensions
        mask = mask.view(1, 1, max_seq_length, max_seq_length)

        # register_buffer saves it as part of the model so it automatically moves to the right device (CPU/GPU)
        self.register_buffer("causal_mask", mask)

    # Splits Q into multiple heads
    def _split_heads(self, x):
        """
        Transforms input embeddings from
        [batch_size, sequence_length, embedding_dim]
        to
        [batch_size, num_heads, sequence_length, head_dim]
        """
        batch_size, sequence_length, embedding_dim = x.shape

        # Split embedding_dim into (num_heads, head_dim)
        x = x.reshape(batch_size, sequence_length, self.num_heads, self.head_dim)

        # Reorder and return the intended shape
        return x.transpose(1, 2)

    # Splits K or V into num_groups heads
    def _split_groups(self, x):
        """
        Transforms K/V from
        [batch_size, sequence_length, num_groups * head_dim]
        to
        [batch_size, num_groups, sequence_length, head_dim]
        """
        batch_size, sequence_length, _ = x.shape

        x = x.reshape(batch_size, sequence_length, self.num_groups, self.head_dim)
        return x.transpose(1, 2)

    # Merge heads back together
    def _merge_heads(self, x):
        """
        Transforms inputs from
        [batch_size, num_heads, sequence_length, head_dim]
        to
        [batch_size, sequence_length, embedding_dim]
        """
        batch_size, num_heads, sequence_length, head_dim = x.shape

        # Move sequence_length back before num_heads in the shape
        x = x.transpose(1, 2)

        # Merge (num_heads, head_dim) back into embedding_dim
        embedding_dim = num_heads * head_dim
        x = x.reshape(batch_size, sequence_length, embedding_dim)

        return x

    # Forward pass
    def forward(self, x):
        batch_size, sequence_length, embedding_dim = x.shape

        # Compute Q, K, V
        Q = self.W_q(x)  # [batch_size, sequence_length, embedding_dim]
        K = self.W_k(x)  # [batch_size, sequence_length, num_groups * head_dim]
        V = self.W_v(x)  # [batch_size, sequence_length, num_groups * head_dim]

        # Split Q into multiple heads
        Q = self._split_heads(Q)  # [batch_size, num_heads, sequence_length, head_dim]

        # Split K and V into num_groups heads
        K = self._split_groups(K)  # [batch_size, num_groups, sequence_length, head_dim]
        V = self._split_groups(V)  # [batch_size, num_groups, sequence_length, head_dim]

        # Expand K and V so each KV group is shared across multiple query heads
        K = K.repeat_interleave(self.group_size, dim=1)  # [batch_size, num_heads, sequence_length, head_dim]
        V = V.repeat_interleave(self.group_size, dim=1)  # [batch_size, num_heads, sequence_length, head_dim]

        # Calculate scaled dot-product attention
        attn_scores = Q @ K.transpose(-2, -1)
        attn_scores = attn_scores / math.sqrt(self.head_dim)

        # Slice the pre-built mask to the current sequence length
        causal_mask = self.causal_mask[:, :, :sequence_length, :sequence_length]

        # Mask out future positions by setting their scores to -inf
        attn_scores = attn_scores.masked_fill(causal_mask == 0, float('-inf'))

        # Apply softmax to get attention weights
        attn_weights = torch.softmax(attn_scores, dim=-1)

        # Multiply attention weights by V
        weighted_values = attn_weights @ V  # [batch_size, num_heads, sequence_length, head_dim]

        # Merge head outputs
        merged_heads_output = self._merge_heads(weighted_values)

        # Obtain final output
        output = self.W_o(merged_heads_output)

        return output

In [ ]:
# Single Expert (Feed-Forward Network)
class Expert(nn.Module):
    def __init__(self, embedding_dim, ff_dim, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(embedding_dim, ff_dim, bias=False)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(ff_dim, embedding_dim, bias=False)

    def forward(self, x):
        x = self.fc1(x) # Expand dimensions to ff_dim
        x = self.activation(x) # GELU activation
        x = self.dropout(x) # Dropout regularization
        x = self.fc2(x) # Project back to embedding_dim

        return x

In [ ]:
# Router
class Router(nn.Module):
    def __init__(self, embedding_dim, num_experts, top_k = 3):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k

        # Linear layer to compute router logits
        self.gate = nn.Linear(embedding_dim, num_experts, bias = False)

    def forward(self, x):
        # Compute logits for each expert
        router_logits = self.gate(x)

        # Select top-k experts with the highest logits
        # top_k_logits: Logit values for top-k selected experts
        # top_k_indices: Indices of top-k selected experts
        top_k_logits, top_k_indices = torch.topk(router_logits, self.top_k, dim=-1)

        # Normalize the top-k scores into weights that sum to 1 using softmax
        top_k_weights = torch.softmax(top_k_logits, dim = -1)

        # For load-balancing later, convert logits to probabilities using softmax over all experts
        router_probs = torch.softmax(router_logits, dim = -1)

        return router_probs, top_k_indices, top_k_weights

In [ ]:
# Mixture-of-Experts layer

class MixtureOfExpertsLayer(nn.Module):
  def __init__(self, embedding_dim, ff_dim, num_experts, top_k, dropout=0.1):
    super().__init__()

    # Each token must pick at least one expert and at most num_experts
    assert 1 <= top_k <= num_experts, "top_k must be between 1 and num_experts"

    self.embedding_dim = embedding_dim
    self.num_experts = num_experts
    self.top_k = top_k

    # Router
    self.router = Router(embedding_dim, num_experts, top_k)

    # Experts
    self.experts = nn.ModuleList([
        Expert(embedding_dim, ff_dim, dropout)
        for _ in range(num_experts)
    ])

  def load_balance_loss(self, router_probs, top_k_indices):
    # Dimensions
    total_tokens, total_experts = router_probs.shape

    # Importance: average router probability for each expert across all tokens
    importance = router_probs.mean(dim=0)

    # Load: fraction of all expert slots assigned to each expert
    all_selected_experts = top_k_indices.reshape(-1)
    load = (
        torch.bincount(all_selected_experts, minlength=total_experts)
        .float() / (total_tokens * self.top_k)
    )

    # Loss encourages uniform distribution across experts
    loss = total_experts * (importance * load).sum()

    return loss

  def forward(self, x):
    batch_size, sequence_length, embedding_dim = x.shape

    # Flatten batch and sequence dimensions to route per token
    num_tokens = batch_size * sequence_length
    x_flat = x.reshape(num_tokens, embedding_dim)

    # Router outputs
    router_probs, top_k_indices, top_k_weights = self.router(x_flat)

    # Initialize output tensor to accumulate expert outputs
    output = torch.zeros_like(x_flat)

    # Process each expert separately (sparse computation)
    for expert_id in range(self.num_experts):
      # Mask to find which tokens selected this expert
      mask = (top_k_indices == expert_id)

      # Skip this expert if no tokens are routed to it
      if not mask.any():
        continue

      # Extract positions where this expert was selected
      # token_ids: which tokens need this expert
      # k_positions: which top-k slot this expert occupies (e.g. 0, 1, or 2 for top-3)
      token_ids, k_positions = mask.nonzero(as_tuple=True)

      # Get embeddings for tokens that selected this expert
      expert_input = x_flat[token_ids]

      # Forward pass through the expert
      expert_output = self.experts[expert_id](expert_input)

      # Get routing weights for this expert's contribution
      weights = top_k_weights[token_ids, k_positions].unsqueeze(-1)

      # Add this expert's weighted output to the final output
      output[token_ids] += expert_output * weights

    # Compute load-balancing auxiliary loss
    aux_loss = self.load_balance_loss(router_probs, top_k_indices)

    # Reshape from (num_tokens, embedding_dim) back to original dimensions
    output = output.reshape(batch_size, sequence_length, embedding_dim)

    return output, aux_loss

In [ ]:
class MixtureOfExpertsDecoder(nn.Module):
    def __init__(self, embedding_dim, ff_dim, num_heads, num_groups,
                 num_experts, top_k, max_seq_length, dropout=0.1):
        super().__init__()

        # Grouped query attention
        self.attention = GroupedQueryAttention(embedding_dim, num_heads, num_groups, max_seq_length)

        # Mixture-of-experts layer
        self.moe_layer = MixtureOfExpertsLayer(
            embedding_dim = embedding_dim,
            ff_dim        = ff_dim,
            num_experts   = num_experts,
            top_k         = top_k,
            dropout       = dropout
        )

        # Pre-LayerNorm
        self.ln1 = nn.LayerNorm(embedding_dim)  # LayerNorm before attention
        self.ln2 = nn.LayerNorm(embedding_dim)  # LayerNorm before MoE layer

        # Dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Self-attention with residual connection
        attn_output = self.attention(self.ln1(x))
        x = x + self.dropout(attn_output)

        # MoE layer with residual connection
        moe_output, aux_loss = self.moe_layer(self.ln2(x))
        x = x + self.dropout(moe_output)

        return x, aux_loss

In [ ]:
class MixtureOfExpertsTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,                  # Total number of tokens in the vocabulary
        embedding_dim,               # Token embedding dimension
        num_heads,                   # Number of attention heads in each decoder
        num_groups,                  # Number of KV groups for GQA
        ff_dim,                      # Hidden dimension of experts
        num_layers,                  # Number of stacked MoE decoders
        max_seq_length,              # Max input sequence length the model can handle
        num_experts       = 8,       # Total number of experts per MoE layer
        top_k             = 2,       # Number of experts to activate per token
        dropout           = 0.1,     # Dropout
        moe_aux_loss_coef = 0.01,    # Coefficient for auxiliary load balancing loss
    ):
        super().__init__()

        self.max_seq_length    = max_seq_length
        self.moe_aux_loss_coef = moe_aux_loss_coef

        # Token embedding layer
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)

        # Learned positional embedding layer
        self.positional_embedding = nn.Embedding(max_seq_length, embedding_dim)

        # Dropout
        self.dropout = nn.Dropout(dropout)

        # Stack of MoE decoders (total: 'num_layers')
        self.decoders = nn.ModuleList([
            MixtureOfExpertsDecoder(
                embedding_dim = embedding_dim,
                ff_dim = ff_dim,
                num_heads = num_heads,
                num_groups = num_groups,
                num_experts = num_experts,
                top_k = top_k,
                max_seq_length = max_seq_length,
                dropout = dropout
            )
            for _ in range(num_layers)
        ])

        # Final layer normalization
        self.final_ln = nn.LayerNorm(embedding_dim)

        # Linear layer to project hidden states to vocabulary size to get logits
        self.output_proj = nn.Linear(embedding_dim, vocab_size, bias=False)

        # Tie output projection weights to token embedding (saves ~12.9M params)
        self.output_proj.weight = self.token_embedding.weight

    def forward(self, x):
        # x is token indices of shape (batch_size, sequence_length)
        batch_size, sequence_length = x.shape

        # Check that sequence length does not exceed maximum
        if sequence_length > self.max_seq_length:
            raise ValueError(f"Input sequence length {sequence_length} exceeds maximum sequence length {self.max_seq_length}.")

        # Create positional indices: [0, 1, 2, ..., sequence_length-1]
        positions = torch.arange(sequence_length, device=x.device)

        # Create token embedding from token indices
        token_embedding = self.token_embedding(x)

        # Create positional embedding from positional indices
        positional_embedding = self.positional_embedding(positions)

        # Combine embeddings and apply dropout
        x = self.dropout(token_embedding + positional_embedding)

        # Array to store auxiliary losses from each MoE decoder
        aux_losses = []

        # Forward pass through MoE decoders sequentially
        for decoder in self.decoders:
            x, aux_loss = decoder(x)
            aux_losses.append(aux_loss)

        # Apply LayerNorm to the output
        x = self.final_ln(x)

        # Project to vocabulary size to get logits for next-token prediction
        logits = self.output_proj(x)

        # Mean auxiliary losses across all MoE decoders and scale by coefficient
        aux_loss = torch.stack(aux_losses).mean() * self.moe_aux_loss_coef

        return logits, aux_loss

In [ ]:
# Set the best available compute device
if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps") # Apple’s MPS backend (for Mac GPUs)
elif torch.cuda.is_available():
    device = torch.device("cuda") # CUDA (for NVIDIA GPUs)
else:
    device = torch.device("cpu") # CPU if none of the above are available

print(f"\nUsing device: {device}\n")

In [ ]:
# Model hyperparameters
EMBEDDING_DIM = 256       # Token embedding dimension
FF_DIM = 1024             # Expert hidden dimension (4 × embedding_dim)
NUM_HEADS = 8             # Number of query heads
NUM_GROUPS = 2            # Number of KV groups for GQA (4 Q heads per group)
NUM_LAYERS = 6            # Number of stacked MoE decoder blocks
NUM_EXPERTS = 8           # Total experts per MoE layer
TOP_K = 2                 # Experts activated per token
DROPOUT = 0.1             # Dropout rate
MOE_AUX_LOSS_COEF = 0.01  # Weight for load balancing loss

In [ ]:
# Initialize model and move it to device
model = MixtureOfExpertsTransformer(
    vocab_size = vocab_size,
    embedding_dim = EMBEDDING_DIM,
    num_heads = NUM_HEADS,
    num_groups = NUM_GROUPS,
    ff_dim = FF_DIM,
    num_layers = NUM_LAYERS,
    max_seq_length = MAX_SEQ_LENGTH,
    num_experts = NUM_EXPERTS,
    top_k = TOP_K,
    dropout = DROPOUT,
    moe_aux_loss_coef = MOE_AUX_LOSS_COEF,
).to(device)

# Convert model into optimized machine code at runtime to speed up training
model = torch.compile(model, dynamic=True)

In [ ]:
# Stats about model parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
print(f"Total parameters:     {total_params:,} ({total_params/1e6:.2f}M)")
print(f"GQA config: {NUM_HEADS} Q heads, {NUM_GROUPS} KV groups, {NUM_HEADS // NUM_GROUPS} Q heads per group")

In [ ]:
# Active parameters per token (MoE activates only top_k of num_experts)
active_params = sum(
    p.numel() * (TOP_K / NUM_EXPERTS) if ".experts." in name else p.numel()
    for name, p in model.named_parameters()
)

print(f"Active parameters: {active_params:,.0f} ({active_params/1e6:.2f}M)")
print(f"Active ratio: {active_params/total_params:.1%}")

In [ ]:
print(model)

In [ ]:
# Training function for one epoch
def train(model, train_loader, optimizer, criterion, device, epoch):
    # Enable training mode
    model.train()

    # Track total loss
    total_loss = 0.0

    # Total number of batches
    num_batches = len(train_loader)

    for input_ids, target_ids in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
        # Move batch to device
        input_ids  = input_ids.to(device)
        target_ids = target_ids.to(device)

        # Forward pass with mixed precision
        with torch.amp.autocast(device_type=device.type, dtype=torch.bfloat16):
            logits, aux_loss = model(input_ids)

            # Calculate cross-entropy loss
            ce_loss = criterion(
                logits.reshape(-1, logits.size(-1)),  # (batch_size * sequence_length, vocab_size)
                target_ids.reshape(-1) # (batch_size * sequence_length)
            )

            # Total loss = cross-entropy loss + load balancing auxiliary loss
            loss = ce_loss + aux_loss

        # Clear old gradients
        optimizer.zero_grad()

        # Standard backward pass
        loss.backward()

        # Clip gradient norms
        clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Update weights
        optimizer.step()

        # Accumulate loss
        total_loss += loss.item()

    # Return epoch average loss
    return total_loss / num_batches

In [ ]:
# Evaluation function
def evaluate(model, val_loader, criterion, device):
    # Enable evaluation mode
    model.eval()

    # Track total loss
    total_loss = 0.0

    # Disable gradient computation
    with torch.no_grad():
        for input_ids, target_ids in val_loader:
            # Move batch to device
            input_ids  = input_ids.to(device)
            target_ids = target_ids.to(device)

            # Forward pass with mixed precision
            with torch.amp.autocast(device_type=device.type, dtype=torch.bfloat16):
                logits, aux_loss = model(input_ids)

                # Calculate cross-entropy loss
                ce_loss = criterion(
                    logits.reshape(-1, logits.size(-1)),  # (batch_size * sequence_length, vocab_size)
                    target_ids.reshape(-1) # (batch_size * sequence_length)
                )

                # Accumulate total loss: cross-entropy loss + load balancing auxiliary loss
                total_loss += (ce_loss + aux_loss).item()

    # Return average loss over all batches
    return total_loss / len(val_loader)

In [ ]:
# Initial learning rate
LEARNING_RATE = 3e-4

# Number of training epochs
TOTAL_EPOCHS = 1

# AdamW optimizer with weight decay for regularization
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# Cross-entropy loss
criterion = nn.CrossEntropyLoss()

# Decay the learning rate following a cosine curve over all epochs
scheduler = CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-5)

In [ ]:
# Train for TOTAL_EPOCHS epochs
for epoch in range(1, TOTAL_EPOCHS + 1):
    # Get current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch}/{TOTAL_EPOCHS}, LR: {current_lr:.6f}")

    # Train and evaluate for this epoch
    avg_train_loss = train(model, train_loader, optimizer, criterion, device, epoch)
    val_loss = evaluate(model, val_loader, criterion, device)

    # Update learning rate
    scheduler.step()

    # Compute perplexity from loss (lower is better)
    train_perplexity = math.exp(avg_train_loss)
    val_perplexity = math.exp(val_loss)

    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"Train Perplexity: {train_perplexity:.2f} | Val Perplexity: {val_perplexity:.2f}")

print("\nTraining complete.")

In [ ]:
# Function to generate text with greedy decoding
def generate_text(model, tokenizer, prompt, max_new_tokens, device):

    # Enable evaluation mode
    model.eval()

    # Encode prompt, convert to tensor and add batch dimension
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    # Disable gradient tracking for inference
    with torch.no_grad():
        for _ in range(max_new_tokens):

            # Keep only the last max_seq_length tokens (context window)
            context_ids = input_ids[:, -model.max_seq_length:]

            # Forward pass through the model
            logits, _ = model(context_ids)

            # Select the most likely next token (greedy decoding)
            next_token = torch.argmax(
                logits[:, -1, :],  # Extract logits from the final time step
                dim=-1,
                keepdim=True
            )

            # Append predicted token to the input sequence
            input_ids = torch.cat([input_ids, next_token], dim=1)

            # Stop if the model generates EOS token
            if next_token.item() == tokenizer.eot_token:
                break

    # Decode token IDs back into text and return
    return tokenizer.decode(input_ids[0].tolist())

In [ ]:
# Test prompts
test_prompts = [
  "Mathematics is",
  "Feynman was",
  "The capital of India"
]

# Create an instance of the untrained model
untrained_model = MixtureOfExpertsTransformer(
    vocab_size = vocab_size,
    embedding_dim = EMBEDDING_DIM,
    num_heads = NUM_HEADS,
    num_groups = NUM_GROUPS,
    ff_dim = FF_DIM,
    num_layers = NUM_LAYERS,
    max_seq_length = MAX_SEQ_LENGTH,
    num_experts = NUM_EXPERTS,
    top_k = TOP_K,
    dropout = DROPOUT,
    moe_aux_loss_coef = MOE_AUX_LOSS_COEF,
).to(device)

# Text generation from untrained model
for prompt in test_prompts:
    generated = generate_text(untrained_model, tokenizer, prompt, max_new_tokens=50, device=device)
    print(f"Prompt: '{prompt}'")
    print(f"Generated: {generated}\n")

In [ ]:
# Text generation from trained model
for prompt in test_prompts:
    generated = generate_text(model, tokenizer, prompt, max_new_tokens=50, device=device)
    print(f"Prompt: '{prompt}'")
    print(f"Generated: {generated}\n")